# Preprocess

Reads `Risk Database - Autonomous AI Risks (all).tsv` into a Polars DataFrame and prints an anomaly report. Running this notebook writes no files and does not change the TSV.

Run the cells from the top. The first code cell loads the table; each later cell prints one check, or `none`.

In [11]:
from pathlib import Path

import polars as pl

TSV = Path("Risk Database - Autonomous AI Risks (all).tsv")

df = pl.read_csv(
    TSV,
    separator="\t",
    schema_overrides={
        "RISK_ID": pl.Int64,
        "Assotiated Ev_ID": pl.String,
        "Risk": pl.String,
        "Comment": pl.String,
    },
)

pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(500)
pl.Config.set_tbl_width_chars(200)

def show(frame: pl.DataFrame) -> None:
    if frame.height == 0:
        print("none")
    else:
        print(frame)

print(f"{df.height} rows")

350 rows


In [12]:
conflicts = (
    df.select("RISK_ID", pl.col("Risk").str.strip_chars().alias("label"))
    .unique()
    .group_by("RISK_ID")
    .agg(pl.col("label").sort())
    .filter(pl.col("label").list.len() > 1)
    .sort("RISK_ID")
)

print("Same RISK_ID, conflicting labels")
show(conflicts)

Same RISK_ID, conflicting labels
shape: (6, 2)
┌─────────┬───────────────────────────────────────────────────────┐
│ RISK_ID ┆ label                                                 │
│ ---     ┆ ---                                                   │
│ i64     ┆ list[str]                                             │
╞═════════╪═══════════════════════════════════════════════════════╡
│ 3       ┆ ["Power Seeking", "Power seeking"]                    │
│ 10      ┆ ["Manipulation (society)", "Modelling (society)"]     │
│ 32      ┆ ["Oversight evasion", "Oversight evasion (systems)"]  │
│ 35      ┆ ["Disinformation", "Oversight evasion"]               │
│ 66      ┆ ["Privacy concerns upon AI use", "Privacy violation"] │
│ 111     ┆ ["Shutdown evasion", "Shutdown resistance"]           │
└─────────┴───────────────────────────────────────────────────────┘


In [13]:
shared = (
    df.select(pl.col("Risk").str.strip_chars().alias("label"), "RISK_ID")
    .unique()
    .group_by("label")
    .agg(pl.col("RISK_ID").sort())
    .filter(pl.col("RISK_ID").list.len() > 1)
    .sort("label")
)

print("Same label, more than one RISK_ID")
show(shared)

Same label, more than one RISK_ID
shape: (4, 2)
┌─────────────────────────────┬───────────┐
│ label                       ┆ RISK_ID   │
│ ---                         ┆ ---       │
│ str                         ┆ list[i64] │
╞═════════════════════════════╪═══════════╡
│ Loss of control             ┆ [26, 115] │
│ Oversight evasion           ┆ [32, 35]  │
│ Oversight evasion (systems) ┆ [32, 54]  │
│ Shutdown resistance         ┆ [90, 111] │
└─────────────────────────────┴───────────┘


In [14]:
duplicates = (
    df.group_by("RISK_ID", "Assotiated Ev_ID")
    .len()
    .filter(pl.col("len") > 1)
    .sort("RISK_ID", "Assotiated Ev_ID")
)

print("Duplicate (RISK_ID, ev_id) pairs")
show(duplicates)

Duplicate (RISK_ID, ev_id) pairs
none


In [15]:
not_coded = df.filter(
    (pl.col("RISK_ID") == 0) | (pl.col("Risk").str.strip_chars() == "NOT CODED")
).select("RISK_ID", "Assotiated Ev_ID", "Risk", "Comment")

print("NOT CODED rows")
show(not_coded)

NOT CODED rows
shape: (4, 4)
┌─────────┬──────────────────┬───────────┬───────────────────────────────────────────────────────────────────────────────────┐
│ RISK_ID ┆ Assotiated Ev_ID ┆ Risk      ┆ Comment                                                                           │
│ ---     ┆ ---              ┆ ---       ┆ ---                                                                               │
│ i64     ┆ str              ┆ str       ┆ str                                                                               │
╞═════════╪══════════════════╪═══════════╪═══════════════════════════════════════════════════════════════════════════════════╡
│ 0       ┆ 22.04.00         ┆ NOT CODED ┆ null                                                                              │
│ 0       ┆ 34.03.00         ┆ NOT CODED ┆ null                                                                              │
│ 0       ┆ 46.03.02         ┆ NOT CODED ┆ null                                   

In [16]:
present = df.filter(pl.col("RISK_ID") != 0).select("RISK_ID").unique()
max_id = present.select(pl.col("RISK_ID").max()).item()
missing = (
    pl.DataFrame({"RISK_ID": range(1, max_id + 1)})
    .join(present, on="RISK_ID", how="anti")
    .sort("RISK_ID")
)

print("Missing RISK_ID values")
show(missing)

Missing RISK_ID values
none


In [17]:
bad_ev_ids = (
    df.filter(
        ~pl.col("Assotiated Ev_ID")
        .str.strip_chars()
        .str.contains(r"^\d+\.\d+\.\d+[A-Za-z]?$")
    )
    .select("Assotiated Ev_ID")
    .unique()
    .sort("Assotiated Ev_ID")
)

print("ev_id values outside digits.digits.digits plus an optional letter")
show(bad_ev_ids)

ev_id values outside digits.digits.digits plus an optional letter
none


In [18]:
whitespace = (
    df.filter(pl.col("Risk") != pl.col("Risk").str.strip_chars())
    .select(
        "RISK_ID",
        "Assotiated Ev_ID",
        pl.col("Risk").map_elements(repr, return_dtype=pl.String),
    )
)

print("Risk labels with leading or trailing whitespace")
show(whitespace)

Risk labels with leading or trailing whitespace
shape: (17, 3)
┌─────────┬──────────────────┬──────────────────────────────────────────────────────────┐
│ RISK_ID ┆ Assotiated Ev_ID ┆ Risk                                                     │
│ ---     ┆ ---              ┆ ---                                                      │
│ i64     ┆ str              ┆ str                                                      │
╞═════════╪══════════════════╪══════════════════════════════════════════════════════════╡
│ 7       ┆ 09.01.01         ┆ 'Ethical decisions regarding termination of human life ' │
│ 17      ┆ 09.04.02         ┆ 'Provide advantage to itself '                           │
│ 46      ┆ 25.05.00         ┆ 'Weapons acquisition  '                                  │
│ 7       ┆ 29.02.03         ┆ 'Ethical decisions regarding termination of human life ' │
│ 61      ┆ 30.02.01         ┆ 'Provision of violent content '                          │
│ 70      ┆ 34.02.01         ┆ 'Gener

In [ ]:
comments = (
    df.select(
        "RISK_ID",
        "Assotiated Ev_ID",
        pl.col("Comment").fill_null("").str.strip_chars().alias("Comment"),
    )
    .filter(pl.col("Comment") != "")
)

print("Non-empty comments")
show(comments)